<a href="https://colab.research.google.com/github/pOrtal0220/Cotton-Prediction-Model/blob/main/Model_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import sys
import os
from collections import OrderedDict

NUM_CLASSES = 4
CLASS_NAMES = ['diseased cotton plant', 'diseased cotton leaf', 'fresh cotton plant', 'fresh cotton leaf']
MODEL_WEIGHTS_PATH = r"/content/modelCottonDemo.pth"

IMAGE_TO_PREDICT = r"/content/dd (16)_iaip.jpg"

try:
    model = models.resnet50(weights=None)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, NUM_CLASSES)
except Exception as e:
    print(f"ERROR: Could not create model architecture. {e}")
    sys.exit()

print(f"Loading weights from '{MODEL_WEIGHTS_PATH}'...")
try:
    if not os.path.exists(MODEL_WEIGHTS_PATH):
        print(f"ERROR: Weight file not found at '{MODEL_WEIGHTS_PATH}'")
        print("Please check the 'MODEL_WEIGHTS_PATH' variable.")
        sys.exit()

    original_state_dict = torch.load(MODEL_WEIGHTS_PATH, map_location=torch.device('cpu'))

    new_state_dict = OrderedDict()

    for key, value in original_state_dict.items():
        if key.startswith('network.'):
            new_key = key.replace('network.', '', 1)
            new_state_dict[new_key] = value
        else:
            new_state_dict[key] = value

    model.load_state_dict(new_state_dict)

    model.eval()

    print("   Successfully loaded and adapted weights!")

except RuntimeError as e:
    print(f"ERROR: Failed to load weights. {e}")
    print("Ensure 'NUM_CLASSES' matches the trained model's output.")
    sys.exit()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    sys.exit()


print("Setting up prediction function...")

data_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict_image(image_path, model):
    try:
        image = Image.open(image_path).convert('RGB')
    except FileNotFoundError:
        print(f"ERROR: Cannot find image at '{image_path}'")
        print("Please check the 'IMAGE_TO_PREDICT' path.")
        return None
    except Exception as e:
        print(f"ERROR: Could not open image. {e}")
        return None

    image_tensor = data_transform(image).unsqueeze(0)
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        _, predicted_idx = torch.max(output, 1)

    predicted_class = CLASS_NAMES[predicted_idx.item()]
    confidence = probabilities[predicted_idx.item()].item() * 100

    return predicted_class, confidence

print("="*30)
print(f"Running prediction for: {IMAGE_TO_PREDICT}")

predicted_result = predict_image(IMAGE_TO_PREDICT, model)

if predicted_result:
    class_name, confidence = predicted_result
    print("\n--- PREDICTION ---")
    print(f"   Class:      {class_name}")
    print(f"   Confidence: {confidence:.2f}%")
    print("="*30)

Loading weights from '/content/modelCottonDemo.pth'...
   Successfully loaded and adapted weights!
Setting up prediction function...
Running prediction for: /content/dd (16)_iaip.jpg

--- PREDICTION ---
   Class:      diseased cotton leaf
   Confidence: 97.35%
